# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [83]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
load_dotenv(override=True)

True

In [84]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [85]:
ai_ml_api_key = os.getenv("AI_ML_API_KEY")
ai_ml = OpenAI(
    base_url="https://api.aimlapi.com/v1",
    api_key=ai_ml_api_key,
)

In [86]:
# Some lists!

todos = []
completed = []

In [87]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [88]:
get_todo_report()

''

In [89]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [90]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [91]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [92]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [93]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [94]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [95]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [96]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [100]:
def loop(messages):
    done = False
    while not done:
        response = ai_ml.chat.completions.create(
            model="openai/gpt-5-2", 
            messages=messages, 
            tools=tools, 
            reasoning_effort="low"
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            
            # Fix: convert message to dict and ensure content is not null
            message_dict = {
                "role": "assistant",
                "content": message.content or "",  # Replace null with empty string
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in tool_calls
                ]
            }
            
            messages.append(message_dict)
            messages.extend(results)
        else:
            done = True
            show(response.choices[0].message.content)

In [98]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [102]:
todos, completed = [], []

messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

loop(messages)

Todo #1: Identify missing quantity (distance between Boston and New York) and choose a reasonable 
estimate/assumption.
Todo #2: Set up relative-motion equation using the head start and closing speed.
Todo #3: Compute meeting time based on the assumed distance and convert to clock time.
Todo #4: Present the result clearly, noting the distance assumption and how the answer would change with a 
different distance.

Missing data identified: The Boston–New York separation isn’t given. Use a common rail/road approximation of about 
215 miles (NYC↔Boston is typically ~215–220 miles depending on route).

Todo #1: Identify missing quantity (distance between Boston and New York) and choose a reasonable 
estimate/assumption.
Todo #2: Set up relative-motion equation using the head start and closing speed.
Todo #3: Compute meeting time based on the assumed distance and convert to clock time.
Todo #4: Present the result clearly, noting the distance assumption and how the answer would change with a 
different distance.

Let t = hours after 2:00 pm when they meet. Then Boston train travels 60t.
NY train leaves at 3:00 pm, so it travels for (t−1) hours and covers 80(t−1).
Meeting condition (sum of distances = total distance D):
60t + 80(t−1) = D.

Todo #1: Identify missing quantity (distance between Boston and New York) and choose a reasonable 
estimate/assumption.
Todo #2: Set up relative-motion equation using the head start and closing speed.
Todo #3: Compute meeting time based on the assumed distance and convert to clock time.
Todo #4: Present the result clearly, noting the distance assumption and how the answer would change with a 
different distance.

Use D = 215 miles.
Solve: 60t + 80(t−1) = 215 → 60t + 80t − 80 = 215 → 140t = 295 → t = 295/140 ≈ 2.1071 hours after 2:00 pm.
0.1071 hr × 60 ≈ 6.43 minutes.
So meeting time ≈ 4:06 pm.

Todo #1: Identify missing quantity (distance between Boston and New York) and choose a reasonable 
estimate/assumption.
Todo #2: Set up relative-motion equation using the head start and closing speed.
Todo #3: Compute meeting time based on the assumed distance and convert to clock time.
Todo #4: Present the result clearly, noting the distance assumption and how the answer would change with a 
different distance.

Provide the computed time and explicitly state it depends on the assumed Boston–NY distance; also give the general 
formula t = (D+80)/140 hours after 2:00 pm.

Todo #1: Identify missing quantity (distance between Boston and New York) and choose a reasonable 
estimate/assumption.
Todo #2: Set up relative-motion equation using the head start and closing speed.
Todo #3: Compute meeting time based on the assumed distance and convert to clock time.
Todo #4: Present the result clearly, noting the distance assumption and how the answer would change with a 
different distance.

They meet at about 4:06 pm (assuming Boston–New York is about 215 miles apart).

Work:
- Let t = hours after 2:00 pm when they meet.
- Boston train distance: 60t
- NY train leaves at 3:00 pm, so it travels (t−1) hours: distance 80(t−1)
- If the cities are D miles apart:  
  60t + 80(t−1) = D  ⇒  140t = D + 80  ⇒  t = (D+80)/140

With D = 215:
- t = (215+80)/140 = 295/140 ≈ 2.107 hours after 2:00 pm  
- 0.107 hr ≈ 6.4 minutes  
- Time ≈ 4:06 pm

If you use a different distance D, plug it into t = (D+80)/140 and add t hours to 2:00 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>